In [ ]:
# Dependencies

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

In [ ]:
df_fa_2 = pd.read_csv('feature_2_sensorhealth.csv')
df_fa_15 = pd.read_csv('feature_15_sensorhealth.csv')
df_fa_22 = pd.read_csv('feature_22_sensorhealth.csv')
df_fa_30 = pd.read_csv('feature_30_sensorhealth.csv')

In [ ]:
def tag_feature_set(df, feature_set_id):
    df = df.copy()
    df["FeatureSet"] = feature_set_id
    # robust count even if Features is a string
    df["NumFeatures"] = df["Features"].apply(lambda s: len(str(s).split(",")))
    return df

df_all = pd.concat([
    tag_feature_set(df_fa_2,  "f2"),
    tag_feature_set(df_fa_15, "f15"),
    tag_feature_set(df_fa_22, "f22"),
    tag_feature_set(df_fa_30, "f30")
], ignore_index=True)


In [ ]:
df_fleet = (
    df_all.groupby(["FeatureSet", "Model"], as_index=False)
          .agg(FP=("FP","sum"), Total=("Total","sum"))
)
df_fleet["FAR"] = df_fleet["FP"] / df_fleet["Total"]*100


In [ ]:
df_fleet_wide = df_fleet.pivot(index="Model", columns="FeatureSet", values="FAR")
df_fleet_wide


In [ ]:
df_T3000 = df_all[df_all["WagonType"] == "T3000"].copy()
df_T3000[["WagonKit", "Model"]].drop_duplicates().sort_values(["WagonKit", "Model"])
df_T3000_agg = (
    df_T3000
    .groupby(["Model", "NumFeatures", "Features"], as_index=False)
    .agg(
        FP_total=("FP", "sum"),
        Total_samples=("Total", "sum")
    )
)

df_T3000_agg["FalseAlarmRate"] = (
    df_T3000_agg["FP_total"] / df_T3000_agg["Total_samples"] * 100
)
df_T3000_ranked = df_T3000_agg.sort_values("FalseAlarmRate")
df_T3000_ranked


In [ ]:
df_4909 = df_all[df_all["WagonType"] == "4909"].copy()
df_4909[["WagonKit", "Model"]].drop_duplicates().sort_values(["WagonKit", "Model"])
df_4909_agg = (
    df_4909
    .groupby(["Model", "NumFeatures", "Features"], as_index=False)
    .agg(
        FP_total=("FP", "sum"),
        Total_samples=("Total", "sum")
    )
)

df_4909_agg["FalseAlarmRate"] = (
    df_4909_agg["FP_total"] / df_4909_agg["Total_samples"] * 100
)
df_4909_ranked = df_4909_agg.sort_values("FalseAlarmRate")
df_4909_ranked

In [ ]:
df_4575 = df_all[df_all["WagonType"] == "4575"].copy()
df_4575[["WagonKit", "Model"]].drop_duplicates().sort_values(["WagonKit", "Model"])
df_4575_agg = (
    df_4575
    .groupby(["Model", "NumFeatures", "Features"], as_index=False)
    .agg(
        FP_total=("FP", "sum"),
        Total_samples=("Total", "sum")
    )
)

df_4575_agg["FalseAlarmRate"] = (
    df_4575_agg["FP_total"] / df_4575_agg["Total_samples"] * 100
)
df_4575_ranked = df_4575_agg.sort_values("FalseAlarmRate")
df_4575_ranked